# §7 体验实验：手算 GRPO 群组优势

这个笔记本只做一件事：同一道数学题，模型给出 **8 个回答**，规则验证器自动判分（对=1，错=0），请你亲手算出每个回答的**优势**（advantage，记号 Â，读作 A-hat）——这个回答比同组平均水平好多少。然后用 NumPy 逐格对账，验证手算没错。

- **三步手算链**：均值 mean → 标准差 std（注意口径：**除以 N，不是 N−1**）→ 优势 Â = (R − mean) / std。
- **为什么值得手算**：GRPO 是 DeepSeek-R1 路线里的关键算法，它砍掉了 PPO 的价值网络，改用**群组相对排名**当基线——同一批回答互相当裁判。这一页纸的算术，就是它全部的心脏。
- **安装说明**：Colab 已预装 NumPy，直接运行即可，无需 pip 单元格。

本笔记本中的代码与输出均来自 2026-10-09 的真实运行记录。

## 1. 为什么要有群组基线

强化学习要回答一个问题：这次回答，比**预期**好多少？好出来的部分才值得强化，差下去的部分要被弱化——这个差值就是**优势**。

麻烦在于"预期"不好定。PPO 的做法是单独养一个**价值网络**（critic）来估预期；GRPO 的做法省钱得多：对同一道题**一次性采一组回答**，直接用这组回答的平均分当预期——**自己组内比高低**。

这样做有两个好处：省掉半个模型的显存；而且奖励换了打分规则（0/1 改成 0/100），组内排名不变，算出来的优势也不变——**对奖励尺度天然稳健**。代价是：全对或全错的组里没有可比对象（第 6 节细说）。

In [ ]:
import numpy as np

# 题面：同一道数学题，模型给出 8 个回答，规则验证器自动判分
# R_i = 1：第 i 个回答答对了；R_i = 0：答错了。奖励来自规则，不来自人的偏好。
R = np.array([1, 0, 1, 0, 0, 1, 0, 0])   # 8 个回答的奖励，照正文表格逐字输入
N = R.size
print(f'{N} 个回答, R = {R.tolist()}, 答对 {int(R.sum())} 个')

## 2. 第一步：手算 mean（先自己算！）

✏️ **拿张纸，先别运行下面的格子**：8 个奖励里有 3 个 1、5 个 0，平均分是多少？写成既约分数和小数两位。

算完再往下看。

In [ ]:
# 答案：mean = 总和 / 个数 = 3/8 = 0.375
# 你的纸算结果应该长这样：3 ÷ 8 = 0.375 ✓
mean = R.sum() / N
print(f'mean = {int(R.sum())}/{N} = {mean:.6f}')

## 3. 第二步：手算 std（口径陷阱：除以 N，不是 N−1）

✏️ **继续手算**：每个回答的奖励减去 mean（0.375），平方，全部加起来，得多少？（提示：3 个 0.625² 加上 5 个 0.375²。）

再除以个数——**注意：这里是除以 N=8，不是 N−1=7**。这是统计学里"总体标准差"与"样本标准差"的分岔口：GRPO 把这 8 个回答当作**完整的组**（全部考察对象），所以除以 N。如果拿计算器按了 ÷(N−1) 的样本口径，会得到 0.517，跟正文表格的 0.484 对不上——那不是正文错了，是口径按错了。

In [ ]:
# 答案：离差平方和 = 3×0.625² + 5×0.375² = 1.171875 + 0.703125 = 1.875
#       方差 = 1.875 / 8 = 0.234375（除以 N，总体口径）
#       std = √0.234375 ≈ 0.484
var = ((R - mean) ** 2).sum() / N        # numpy 默认 ddof=0，正是总体口径
std = np.sqrt(var)
print(f'方差 = 1.875/{N} = {var:.6f},  std(总体, ÷N) = {std:.6f}')
print(f'对照：样本 std(÷N−1) = {R.std(ddof=1):.6f} —— 正文不采此口径，对不上表')

## 4. 第三步：手算 Â，逐格填表

优势公式照正文抄：**Â_i = (R_i − mean) / std**。

✏️ **把下表填完再运行**：答对的格子 Â = 0.625 ÷ 0.484 ≈ ______；答错的格子 Â = −0.375 ÷ 0.484 ≈ ______（保留两位小数）。

| 样本 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|---|
| R | 1 | 0 | 1 | 0 | 0 | 1 | 0 | 0 |
| Â |  |  |  |  |  |  |  |  |

In [ ]:
A = (R - mean) / std                    # 逐格：答对的 (1−0.375)/0.484 ≈ +1.29，答错的 ≈ −0.77
print(f'{"样本":<6}{"R":>3}{"R−mean":>10}{"Â":>8}')
print('-' * 29)
for i in range(N):
    print(f'{i+1:<6}{R[i]:>3}{R[i]-mean:>10.3f}{A[i]:>8.2f}')
print('-' * 29)

## 5. 与正文表格对账（断言格）

正文表格给出的答案是 mean=3/8=0.375、std=0.484，Â 两档取值 **+1.29**（答对）与 **−0.77**（答错）。下面让计算机当裁判：手算链条的全部数字与正文逐格一致才算 PASS（容差 0.01，应对四舍五入）。

In [ ]:
expected = np.array([1.29, -0.77, 1.29, -0.77, -0.77, 1.29, -0.77, -0.77])
assert abs(mean - 0.375) < 1e-9, mean
assert abs(std - 0.4841) < 1e-3, std
assert np.all(np.abs(np.round(A, 2) - expected) < 0.01), A
print('对账: mean/std/Â 与正文表格逐格一致 → PASS')

## 6. 边界：全对组或全错组，std = 0，优势无定义

如果 8 个回答**全对**（或全错），每个奖励都等于 mean，离差全为 0，std = 0。这时候 Â = 0/0，没有定义——NumPy 会给出 nan（not a number）。

这不是程序的毛病，而是算法的诚实：**组内没有可比对象**，谁更好、谁更坏无从谈起，GRPO 对这一组干脆不下手（工业实现里会换一批题重采）。想一想：全对组里每个回答都该被强化吗？按"比组内平均好"的规矩，它们一样好——没有信号。

In [ ]:
for name, R0 in [('全对组', np.ones(8)), ('全错组', np.zeros(8))]:
    s0 = R0.std()                        # 全同奖励，标准差为 0
    with np.errstate(invalid='ignore'):  # 0/0 = nan 是预期结果，屏蔽告警
        A0 = (R0 - R0.mean()) / s0       # 无定义：nan
    print(f'{name}: std = {s0:.1f}, Â = {A0[0]}（nan = 无定义，跳过该组）')

## 7. 要点回顾

- **组内比高低，不比绝对值**：优势是回答与**同组平均分**的差（再除以 std），不是奖励的绝对大小。把 0/1 改标成 0/100，mean 和 std 同比例放大，Â 一格不变——这就是"对奖励尺度天然稳健"的算术真相。
- **std 口径：除以 N，不是 N−1**。计算器按出 0.517 的读者，是落进了样本标准差的按钮；正文与本书一律用总体口径（0.484）。
- **正优势被强化，负优势被弱化**：答对的 +1.29 告诉训练器"往这个方向更新参数"，答错的 −0.77 告诉它"反着来"。GRPO 不需要绝对奖励值——只需要群组相对排名。
- **全对/全错组没有信号**：std=0 时优势无定义，这是组内比较框架自带的边界，不是 bug。
- 实测打印原文、机器环境与判定结果见同目录《run_log.txt》。